# DuckPD: Temporal Semantics, Fixed-Duration Rolling Windows & Categoricals

This notebook demonstrates the newly added features from the recent releases and changelog:
1. **Out-of-Core Data Ingestion & Setup**: Preserving file scan order and configuring session-level execution.
2. **Fixed-Duration Rolling Windows (`DataFrame.rolling` & `GroupBy.rolling`)**: Nanosecond `RANGE` frames with string/timedelta durations, boundary options (`closed='right'|'left'|'both'|'neither'`), and pandas-compatible defaults.
3. **Lazy Temporal Accessors & Arithmetic (`Series.dt`)**: Fixed-duration `floor()`, `ceil()`, and `round()`, timezone conversion (`tz_convert`), UTC localization (`tz_localize`), and timestamp/duration arithmetic (`+`, `-`, comparisons).
4. **Categorical Semantics (`Series.cat`) & Unused-Category Groupbys**: Categorical metadata preservation, `cat.codes`, `as_ordered()`, ordered comparisons, and `groupby(observed=False)` aggregation expansions.
5. **Direct SQL Execution & Lazy Expression Chaining**: Seamless blending between DuckPD DataFrames, DuckDB SQL queries, and logical plan inspection.
6. **Zero-Copy Arrow & Pandas Interoperability**: Collecting results into Pandas and PyArrow structures while verifying semantic correctness and execution counts.

## 1. Environment Setup and DuckPD Initialization

We initialize a DuckPD session, inspect session configurations, and verify zero eager executions during registration.

In [4]:
import datetime as dt
from datetime import timedelta
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd
import pyarrow as pa
from pandas.testing import assert_frame_equal, assert_series_equal

import duckpd

# Connect to a DuckPD session
session = duckpd.connect()
print(f"DuckPD version: {duckpd.__version__}")
print(f"Initial session execution count: {session.execution_count}")

DuckPD version: 0.1.4
Initial session execution count: 0


## 2. Out-of-Core Data Ingestion (Parquet and CSV)

DuckPD scans Parquet and CSV files lazily without loading entire datasets into memory. Starting in recent versions, CSV and Parquet readers preserve file scan order automatically via hidden stable row identities.

Let's generate a time-series dataset of financial orders across multiple asset classes and write it to Parquet and CSV.

In [5]:
temp_dir = TemporaryDirectory()
work_dir = Path(temp_dir.name)
parquet_path = work_dir / "market_orders.parquet"
csv_path = work_dir / "market_orders.csv"

# Construct a realistic multi-asset trade & order dataset with timestamps and categories
base_time = pd.Timestamp("2024-03-10 09:30:00", tz="UTC")
timestamps = pd.Series([base_time + pd.Timedelta(minutes=5 * i) for i in range(12)], dtype="datetime64[ns, UTC]")

raw_df = pd.DataFrame(
    {
        "order_id": [f"ORD_{1000 + i}" for i in range(12)],
        "timestamp": timestamps,
        "symbol": ["AAPL", "NVDA", "AAPL", "GOOG", "NVDA", "AAPL", "GOOG", "AAPL", "NVDA", "GOOG", "AAPL", "NVDA"],
        "asset_class": pd.Categorical(
            ["Equity", "Equity", "Equity", "Equity", "Equity", "Equity", "Equity", "Equity", "Equity", "Equity", "Equity", "Equity"],
            categories=["Equity", "Fixed_Income", "Commodity", "Crypto"],
            ordered=True,
        ),
        "execution_venue": pd.Categorical(
            ["NASDAQ", "NASDAQ", "NASDAQ", "BATS", "NASDAQ", "ARCA", "BATS", "NASDAQ", "ARCA", "BATS", "ARCA", "NASDAQ"],
            categories=["NASDAQ", "ARCA", "BATS", "IEX", "DARK_POOL"],
            ordered=False,
        ),
        "price": [182.50, 895.00, 183.10, 142.20, 899.50, 182.90, 143.00, 184.00, 902.10, 142.75, 183.80, 905.00],
        "quantity": [150, 40, 200, 300, 50, 180, 250, 350, 60, 400, 220, 75],
        "latency_ms": [12, 18, 9, 25, 15, 11, 30, 8, 14, 28, 10, 16],
    }
)

# Write to disk
raw_df.to_parquet(parquet_path, index=False)
raw_df.to_csv(csv_path, index=False)

# Lazily scan parquet and csv out-of-core
orders_pq = session.read_parquet(parquet_path, order_by="timestamp")
orders_csv = session.read_csv(csv_path)

# Retain pandas source snapshot with full categorical metadata and ordering
orders = session.from_pandas(raw_df, order_by="timestamp")

print(f"Scanned Parquet frame columns: {orders_pq.columns}")
print(f"Scanned CSV frame columns: {orders_csv.columns}")
print(f"Orders frame columns: {orders.columns}")
print(f"Eager executions so far (should remain 0): {session.execution_count}")

Scanned Parquet frame columns: ('order_id', 'timestamp', 'symbol', 'asset_class', 'execution_venue', 'price', 'quantity', 'latency_ms')
Scanned CSV frame columns: ('order_id', 'timestamp', 'symbol', 'asset_class', 'execution_venue', 'price', 'quantity', 'latency_ms')
Orders frame columns: ('order_id', 'timestamp', 'symbol', 'asset_class', 'execution_venue', 'price', 'quantity', 'latency_ms')
Eager executions so far (should remain 0): 0


## 3. Lazy Filtering and Expression Chaining

DuckPD builds an optimized logical plan without executing until required (`head()`, `collect()`, `write_parquet()`, etc.).
Here we showcase:
- **Timestamp and Duration Arithmetic**: Adding/subtracting `timedelta` objects directly on Series.
- **Fixed-Duration Temporal Rounding**: `Series.dt.floor()`, `ceil()`, and `round()` using positive duration frequencies (e.g., `'15min'`, `'1h'`).
- **Timezone Operations**: `tz_convert('America/New_York')` for timezone awareness, and `tz_localize(None)` for stripping timezone metadata.
- **Categorical Accessors & Ordered Filtering**: Checking `.cat.categories`, `.cat.codes`, and comparing ordered categoricals.

In [6]:
# 1. Inspect categorical metadata directly on lazy Series
print("Venue categories:", orders["execution_venue"].cat.categories.tolist())
print("Venue is ordered:", orders["execution_venue"].cat.ordered)
print("Asset class is ordered:", orders["asset_class"].cat.ordered)

# 2. Chain expressions: temporal arithmetic, rounding, tz conversion, and categorical codes
transformed = (
    orders[orders["price"] > 150.0]
    .assign(
        # Duration arithmetic: project trade settlement timestamp (+2 hours)
        settlement_time=lambda df: df["timestamp"] + timedelta(hours=2),
        # Fixed duration rounding: bucket into 15-minute intervals
        time_15m_floor=lambda df: df["timestamp"].dt.floor("15min"),
        time_15m_ceil=lambda df: df["timestamp"].dt.ceil("15min"),
        time_1h_round=lambda df: df["timestamp"].dt.round("1h"),
        # Timezone conversion: convert UTC to Eastern Wall Clock
        ny_time=lambda df: df["timestamp"].dt.tz_convert("America/New_York"),
        # Extract integer category codes lazily
        venue_code=lambda df: df["execution_venue"].cat.codes,
        # Dollar trade notional value
        notional=lambda df: df["price"] * df["quantity"],
    )
)

print(f"Executions before preview/collect: {session.execution_count}")

# Preview top 5 rows using head()
preview = transformed[["order_id", "symbol", "timestamp", "ny_time", "time_15m_floor", "notional", "venue_code"]].head(5)
preview

Venue categories: ['NASDAQ', 'ARCA', 'BATS', 'IEX', 'DARK_POOL']
Venue is ordered: False
Asset class is ordered: True
Executions before preview/collect: 0


,order_id,timestamp,symbol,time_15m_floor,ny_time,venue_code,notional
0,ORD_1000,2024-03-10 09:30:00+00:00,AAPL,2024-03-10 09:30:00+00:00,2024-03-10 05:30:00-04:00,0,27375.0
1,ORD_1001,2024-03-10 09:35:00+00:00,NVDA,2024-03-10 09:30:00+00:00,2024-03-10 05:35:00-04:00,0,35800.0
2,ORD_1002,2024-03-10 09:40:00+00:00,AAPL,2024-03-10 09:30:00+00:00,2024-03-10 05:40:00-04:00,0,36620.0
3,ORD_1004,2024-03-10 09:50:00+00:00,NVDA,2024-03-10 09:45:00+00:00,2024-03-10 05:50:00-04:00,0,44975.0
4,ORD_1005,2024-03-10 09:55:00+00:00,AAPL,2024-03-10 09:45:00+00:00,2024-03-10 05:55:00-04:00,1,32922.0


In [7]:
# Inspect the optimized logical plan and compiled DuckDB SQL for the transformed frame
print(transformed.explain())

Fallback boundaries: none (policy=error)
Materialization boundaries: none in the logical plan
Remote source boundaries: none
Source fragments: []
Cross-source movement: []
DuckPD logical plan:
{
  "node": "ProjectPlan",
  "input": {
    "node": "ProjectPlan",
    "input": {
      "node": "ProjectPlan",
      "input": {
        "node": "ProjectPlan",
        "input": {
          "node": "ProjectPlan",
          "input": {
            "node": "ProjectPlan",
            "input": {
              "node": "ProjectPlan",
              "input": {
                "node": "FilterPlan",
                "input": {
                  "node": "SortPlan",
                  "input": {
                    "node": "ScanPlan",
                    "source": {
                      "node": "PandasSource",
                      "key": "8470658046e54d09becd42365be0233e"
                    },
                    "metadata": {
                      "node": "FrameMetadata",
                      "columns": [
  

## 4. Advanced Window Functions and Grouped Aggregations

Recent DuckPD updates introduced:
1. **Fixed-Duration Rolling Windows**: `DataFrame.rolling("30min", on="timestamp")` and `DataFrameGroupBy.rolling(...)` compiling to nanosecond `RANGE` frames. Supports boundary semantics (`closed='right'|'left'|'both'|'neither'`) and `min_periods`.
2. **Categorical `groupby(observed=False)` Aggregations**: Automatically expanding unused categories in categorical group keys (e.g. keeping unrepresented venues or asset classes in the summary output).

In [10]:
# 1. Global fixed-duration rolling 20-minute window with closed='right'
rolling_global = orders.rolling("20min", on="timestamp", closed="right").mean(numeric_only=True)
print("Global 20-minute rolling means (numeric columns):")
print(rolling_global.collect().head(6))

# 2. Per-symbol fixed-duration rolling 30-minute window
# Group keys compile to window partitions, and timestamps define the range frame
rolling_by_symbol = (
    orders
    .groupby("symbol")
    .rolling("30min", on="timestamp", closed="right")
    .mean(numeric_only=True)
)
print("\nPer-symbol 30-minute rolling means:")
rolling_by_symbol.collect()

Global 20-minute rolling means (numeric columns):
                  timestamp    price  quantity  latency_ms
0 2024-03-10 09:30:00+00:00  182.500     150.0       12.00
1 2024-03-10 09:35:00+00:00  538.750      95.0       15.00
2 2024-03-10 09:40:00+00:00  420.200     130.0       13.00
3 2024-03-10 09:45:00+00:00  350.700     172.5       16.00
4 2024-03-10 09:50:00+00:00  529.950     147.5       16.75
5 2024-03-10 09:55:00+00:00  351.925     182.5       15.00

Per-symbol 30-minute rolling means:


timestamp       price    quantity  latency_ms
symbol                                                                 
AAPL   0  2024-03-10 09:30:00+00:00  182.500000  150.000000   12.000000
       2  2024-03-10 09:40:00+00:00  182.800000  175.000000   10.500000
       5  2024-03-10 09:55:00+00:00  182.833333  176.666667   10.666667
       7  2024-03-10 10:05:00+00:00  183.333333  243.333333    9.333333
       10 2024-03-10 10:20:00+00:00  183.566667  250.000000    9.666667
GOOG   3  2024-03-10 09:45:00+00:00  142.200000  300.000000   25.000000
       6  2024-03-10 10:00:00+00:00  142.600000  275.000000   27.500000
       9  2024-03-10 10:15:00+00:00  142.875000  325.000000   29.000000
NVDA   1  2024-03-10 09:35:00+00:00  895.000000   40.000000   18.000000
       4  2024-03-10 09:50:00+00:00  897.250000   45.000000   16.500000
       8  2024-03-10 10:10:00+00:00  900.800000   55.000000   14.500000
       11 2024-03-10 10:25:00+00:00  903.550000   67.500000   15.000000

In [11]:
# Categorical Groupby with observed=False
# This preserves and expands all declared categories in the output, even if no rows exist for them!
venue_summary = (
    orders
    .groupby("execution_venue", observed=False, as_index=True)
    .agg(
        order_count=("quantity", "count"),
        total_shares=("quantity", "sum"),
        avg_price=("price", "mean"),
    )
)

print("Venue aggregation with observed=False (notice IEX and DARK_POOL are preserved with nulls/zeros):")
venue_df = venue_summary.collect()
venue_df

Venue aggregation with observed=False (notice IEX and DARK_POOL are preserved with nulls/zeros):


,order_count,total_shares,avg_price
execution_venue,,,
NASDAQ,6,865,541.516667
ARCA,3,460,422.933333
BATS,3,950,142.650000
IEX,0,0,NaN
DARK_POOL,0,0,NaN


## 5. Direct SQL Execution on DuckPD DataFrames

You can run raw DuckDB SQL queries directly against existing DuckPD DataFrame objects using `session.sql()`. DuckPD registers or references the underlying relations seamlessly, allowing you to interleave SQL analytics and DataFrame API transformations.

In [12]:
# Run DuckDB SQL directly on the Parquet dataset using session.sql()
sql_query = f"""
    SELECT
        symbol,
        count(*) AS trade_count,
        round(avg(price), 2) AS vwap,
        min(timestamp) AS first_trade,
        max(timestamp) AS last_trade
    FROM read_parquet('{parquet_path}')
    GROUP BY symbol
    ORDER BY trade_count DESC, symbol ASC
"""

sql_frame = session.sql(sql_query)
print("Result of direct DuckDB SQL query:")
sql_result = sql_frame.collect()
sql_result

Result of direct DuckDB SQL query:


,symbol,trade_count,vwap,first_trade,last_trade
0,AAPL,5,183.26,2024-03-10 09:30:00+00:00,2024-03-10 10:20:00+00:00
1,NVDA,4,900.40,2024-03-10 09:35:00+00:00,2024-03-10 10:25:00+00:00
2,GOOG,3,142.65,2024-03-10 09:45:00+00:00,2024-03-10 10:15:00+00:00


## 6. Zero-Copy Conversion to Apache Arrow and Pandas

DuckPD integrates smoothly with downstream ecosystems:
- Collect results as standard `pandas.DataFrame` or `pandas.Series` with exact categorical dtypes and timezone-aware timestamps preserved.
- Collect zero-copy `pyarrow.Table` objects via `to_arrow()`.
- Export directly to Parquet (`write_parquet()`) or CSV (`write_csv()`) directly inside DuckDB without materializing full intermediate tables in Python memory.

In [13]:
# 1. Zero-copy export to PyArrow Table
arrow_table = transformed.to_arrow()
print(f"Exported PyArrow Table type: {type(arrow_table)}")
print(f"Schema:\n{arrow_table.schema}")
print(f"PyArrow Row Count: {arrow_table.num_rows}")

# 2. Collect as Pandas DataFrame and verify categorical and temporal metadata
collected_pandas = transformed.collect()
print("\nPandas DataFrame dtypes:")
print(collected_pandas.dtypes)

# Verify categorical metadata round-trip
assert isinstance(collected_pandas["execution_venue"].dtype, pd.CategoricalDtype)
assert list(collected_pandas["execution_venue"].cat.categories) == ["NASDAQ", "ARCA", "BATS", "IEX", "DARK_POOL"]

# 3. Direct out-of-core write to Parquet
output_parquet = work_dir / "processed_market_orders.parquet"
transformed.write_parquet(output_parquet)
print(f"\nWritten out-of-core Parquet file size: {output_parquet.stat().st_size} bytes")

# Clean up temporary directory
temp_dir.cleanup()
print("\nCompleted successfully! Final session execution count:", session.execution_count)

Exported PyArrow Table type: <class 'pyarrow.lib.Table'>
Schema:
order_id: string
timestamp: timestamp[us, tz=Etc/UTC]
symbol: string
asset_class: dictionary<values=string, indices=uint8, ordered=0>
execution_venue: dictionary<values=string, indices=uint8, ordered=0>
price: double
quantity: int64
latency_ms: int64
settlement_time: timestamp[us, tz=Etc/UTC]
time_15m_floor: timestamp[us, tz=Etc/UTC]
time_15m_ceil: timestamp[us, tz=Etc/UTC]
time_1h_round: timestamp[us, tz=Etc/UTC]
ny_time: timestamp[us, tz=Etc/UTC]
venue_code: int8
notional: double
PyArrow Row Count: 9

Pandas DataFrame dtypes:
order_id                                        str
timestamp                       datetime64[ns, UTC]
symbol                                          str
asset_class                                category
execution_venue                            category
price                                       float64
quantity                                      int64
latency_ms                           